# Velocity-resolved light curves (per velocity bin)

Same measurement approach as `spec_lc_pipeline`, but the line flux is
integrated **per velocity bin** instead of over the whole `LINE_WINDOW`.
One light curve per bin → ICCF each against the 5100 Å continuum → lag vs
velocity (Denney et al. 2009; MAHA I §4.4).

**Inputs** (run these first):
1. `subtract_narrow_mean` → `spec_mean/rms_<line>.txt`, `<obj>_narrow_profile.txt`
2. `velocity_bins` → `<obj>_velbins_<tag>_<scheme>.txt` (the **edge table** read here)
3. same per-epoch spectra + `flux.lst` as `spec_lc_pipeline`

**Per epoch, per bin** (mirrors `spec_lc_pipeline` steps 1b–5):
- subtract the fixed narrow model (**full model here**, `SUBTRACT_NARROW_HB = True`)
- subtract the linear continuum through the `CONT_BLUE`/`CONT_RED` anchor medians
- Simpson-integrate the bin **edge to edge** (grid augmented at the exact edges)
- errors in quadrature: spectral `sqrt(Σerr²)·dwave` ⊕ same-night exposure
  scatter ⊕ [OIII]-scaling `syserr·flux` ⊕ per-bin narrow-model MC constant

**Outputs**: `lc_<line>_<tag>_<scheme>_bin<k>.txt` (3 columns `JD flux err`,
ready for CCF/MICA), a summary PDF, and a consistency check (sum of bins vs
direct whole-window integration, and vs `lc_<line>.txt`).

> **Narrow-Hβ convention differs from the integrated light curve**: here the
> full narrow model ([OIII]4959,5007 **+ narrow Hβ**) is subtracted — the bins
> are broad-line-only. A leftover narrow Hβ would sit as a constant in the
> core bins: CCF-neutral, but it dilutes their fractional variability and the
> absolute bin fluxes. The integrated `lc_<line>.txt` keeps narrow Hβ (MAHA
> convention), so sum(bins)/`lc_<line>` shows a constant offset — the ratio
> must still be flat in time.


In [ ]:
import os
import ast
import re
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import simpson
from scipy.stats import linregress

# =====================================================================
# CONFIGURATION  --  edit this block for a new target / bin scheme
# =====================================================================

TARGET    = "NGC3227"          # used for plot titles only
LINE_NAME = "hbeta"            # label for output file names

# Velocity-bin edge table written by velocity_bins (primary: rms + equal
# flux; swap for the _eqwidth or _mean_ variants to compare schemes).
VELBINS_FILE = "NGC3227_velbins_rms_eqflux.txt"

# Same inputs as spec_lc_pipeline ----------------------------------------
INPUT_DIR = "/Users/kaiwenzhang/Desktop/NGC 3227/Halpha_analysis/agn_only_dered_spectra"
EXPOSURE_DIR = INPUT_DIR
FLUX_LST = "/Users/kaiwenzhang/Desktop/NGC 3227/NGC3227/flux.lst"
RAW_COMBINED_DIR = os.path.dirname(FLUX_LST)
OUTPUT_DIR = os.getcwd()

COMBINED_PATTERN = "_combined.txt"   # marks a per-epoch combined spectrum
EXPOSURE_PATTERN = "fits.cali.txt"   # marks an individual per-night exposure
EXCLUDE_SUBSTR = ["rebin", ".out", "20200205"]   # 20200205 = bad night (NGC3227)

# Continuum anchors (rest-frame A) -- identical to spec_lc_pipeline so the
# bin fluxes are consistent with the integrated line light curve.
CONT_BLUE = (4780.0, 4800.0)
CONT_RED  = (4950.0, 4970.0)

# Narrow-line handling. Unlike the integrated light curve (MAHA convention,
# narrow Hbeta kept), the velocity bins remove the FULL narrow model --
# narrow Hbeta would pile a constant into the core bins and dilute their
# fractional variability.
NARROW_PROFILE = "NGC3227_narrow_profile.txt"
SUBTRACT_NARROW_HB = True    # True = [OIII] + narrow Hbeta: broad-only bins

plt.rcParams.update({"figure.dpi": 110, "font.size": 12})
print(f"Target: {TARGET} | line: {LINE_NAME}")
print(f"Bins  : {VELBINS_FILE}")
print(f"Narrow profile: {NARROW_PROFILE} | subtract narrow Hbeta: {SUBTRACT_NARROW_HB}")


### Helpers — verbatim from `spec_lc_pipeline` (loader, epoch list, syserr, used exposures)

In [ ]:
def load_spectrum(path):
    """Return (wavelength, flux, err) sorted by wavelength.

    Robust to either (N, 3) column layout or (3, N) row layout.
    """
    arr = np.loadtxt(path)
    if arr.ndim != 2:
        raise ValueError(f"Spectrum {path} is not 2-D (shape {arr.shape}).")
    if arr.shape[1] == 3:        # (N, 3): columns are wl, flux, err
        lam, flux, err = arr.T
    elif arr.shape[0] == 3:      # (3, N): rows are wl, flux, err
        lam, flux, err = arr
    else:
        raise ValueError(f"Spectrum {path} is not 3-column (shape {arr.shape}).")
    order = np.argsort(lam)
    return lam[order], flux[order], err[order]


def _excluded(name):
    return any(s in name for s in EXCLUDE_SUBSTR)


def read_used_exposures(combined_name):
    """List the raw exposures that were stacked into a combined spectrum."""
    path = os.path.join(RAW_COMBINED_DIR, combined_name)
    if not os.path.exists(path):
        return None
    with open(path) as f:
        for line in f:
            if not line.startswith("#"):      # header sits above the data rows
                break
            if "combined spec:" in line:
                payload = line.split("combined spec:", 1)[1].strip()
                try:
                    return list(ast.literal_eval(payload))
                except (ValueError, SyntaxError):
                    return None
    return None


def read_epochs(flux_lst):
    """Read (name, jd) for the combined epochs, sorted by JD."""
    names, jds = [], []
    with open(flux_lst) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 2:
                continue
            name = parts[0]
            if COMBINED_PATTERN not in name or _excluded(name):
                continue
            try:
                jd = float(parts[1])
            except ValueError:        # header / non-numeric line
                continue
            if not os.path.exists(os.path.join(INPUT_DIR, name)):
                print(f"  [skip] {name}: not found in INPUT_DIR")
                continue
            names.append(name)
            jds.append(jd)
    order = np.argsort(jds)
    names = [names[i] for i in order]
    jds = np.asarray([jds[i] for i in order], dtype=float)
    return names, jds


def read_syserr(combined_name):
    """Fractional systematic error of a combined epoch (see spec_lc_pipeline)."""
    path = os.path.join(RAW_COMBINED_DIR, combined_name)
    if not os.path.exists(path):
        return 0.0
    with open(path) as f:
        for line in f:
            if not line.startswith("#"):       # header sits above the data rows
                break
            if "syserr:" in line:
                try:
                    payload = line.split("syserr:", 1)[1]
                    vals = [float(x) for x in
                            payload.split("[", 1)[1].split("]", 1)[0].split()]
                except (IndexError, ValueError):
                    return 0.0
                if len(vals) > 1:
                    return np.std(vals, ddof=1) / np.sqrt(len(vals))
                return 0.0
    return 0.0


epoch_names, jd = read_epochs(FLUX_LST)
syserr = np.array([read_syserr(name) for name in epoch_names])
print(f"{len(epoch_names)} epochs | JD {jd.min():.2f} -> {jd.max():.2f}")
print(f"syserr parsed: {np.count_nonzero(syserr > 0)}/{len(syserr)} epochs nonzero")


## Step 1 — Read the velocity-bin edge table

Wavelength edges come from `velocity_bins` (columns `lam_lo_A`, `lam_hi_A`); bins are checked contiguous. `flag` marks bins narrower than the LSF or with multi-crossing edges — carried into the output headers so downstream lag analysis can weigh them.

In [ ]:
vb = np.atleast_2d(np.loadtxt(VELBINS_FILE, usecols=range(9)))
bin_vlo, bin_vhi = vb[:, 1], vb[:, 2]
bin_lo, bin_hi = vb[:, 4], vb[:, 5]
bin_vcen = vb[:, 6]
nbins = vb.shape[0]

bin_flags, meta = [], ""
with open(VELBINS_FILE) as f:
    for line in f:
        if line.startswith("#"):
            if "scheme" in line:
                meta = line.lstrip("# ").strip()
            continue
        if line.strip():
            bin_flags.append(line.split()[-1])

m_scheme = re.search(r"scheme (\w+)", meta)
m_lam0 = re.search(r"lam0 ([0-9.]+)", meta)
SCHEME = m_scheme.group(1) if m_scheme else "bins"
LAM0 = float(m_lam0.group(1)) if m_lam0 else np.nan
base = os.path.basename(VELBINS_FILE).lower()
TAG_SPEC = "rms" if "rms" in base else "mean" if "mean" in base else "spec"

if not np.allclose(bin_hi[:-1], bin_lo[1:]):
    raise ValueError("bins are not contiguous -- edge table corrupted?")
print(meta)
print(f"{nbins} bins ({SCHEME}, {TAG_SPEC} spectrum), "
      f"window {bin_lo[0]:.1f}-{bin_hi[-1]:.1f} A, v=0 at {LAM0:.4f} A")
for k in range(nbins):
    print(f"  bin {k+1:2d}  v {bin_vlo[k]:8.1f} .. {bin_vhi[k]:8.1f} km/s  "
          f"lam {bin_lo[k]:8.3f}-{bin_hi[k]:8.3f}  flag {bin_flags[k]}")


## Step 2 — Narrow model + per-bin measurement functions

Identical physics to `spec_lc_pipeline` Steps 1b–3: fixed narrow model subtracted, linear continuum through the anchor-window medians, Simpson integration — only the integration limits are now the bin edges. Each bin is integrated **edge to edge** on a grid augmented with the interpolated flux at the exact edges (bins tile the window; pixel-only integration would drop the strips between an edge and the outermost pixel — tens of percent for ~5-pixel bins). The error sum uses half-open pixel masks (`[lo, hi)`, last bin closed) so every pixel's error counts exactly once; the narrow-model MC error is integrated per bin (fully-correlated bound, constant across epochs).

In [ ]:
if NARROW_PROFILE is not None and os.path.exists(NARROW_PROFILE):
    prof = np.loadtxt(NARROW_PROFILE)
    # columns: wave  narrow_total sig  oiii5007 sig  oiii4959 sig  narrow_hb sig
    nw = prof[:, 0]
    if SUBTRACT_NARROW_HB:
        nf, nsig = prof[:, 1], prof[:, 2]
        print("narrow model: [OIII]4959,5007 + narrow Hbeta -> broad-only bins")
    else:
        nf = prof[:, 3] + prof[:, 5]          # [OIII] doublet only
        nsig = prof[:, 4] + prof[:, 6]        # fully-correlated bound
        print("narrow model: [OIII]4959,5007 only -> narrow Hbeta stays in "
              "(MAHA LC convention)")

    def subtract_narrow(lam, flux):
        """Remove the fixed narrow model (interpolated; zero outside its span)."""
        return flux - np.interp(lam, nw, nf, left=0.0, right=0.0)

    # Per-bin constant systematic from the narrow model (same for every
    # epoch; widens absolute-flux errors, adds no variability).
    narrow_bin_err = np.zeros(nbins)
    for k in range(nbins):
        m = (nw >= bin_lo[k]) & (nw <= bin_hi[k])
        if m.sum() > 1:
            narrow_bin_err[k] = simpson(nsig[m], nw[m])
    print("per-bin narrow-model error:",
          " ".join(f"{e:.2e}" for e in narrow_bin_err))
else:
    if NARROW_PROFILE is not None:
        print(f"[WARN] NARROW_PROFILE not found: {NARROW_PROFILE} -> "
              "narrow subtraction OFF")

    def subtract_narrow(lam, flux):
        return flux

    narrow_bin_err = np.zeros(nbins)


def line_profile(lam, flux):
    """Narrow-subtracted, continuum-subtracted flux (full array).

    Continuum = straight line through the CONT_BLUE / CONT_RED window
    medians, same anchors as spec_lc_pipeline's subtract_local_continuum.
    """
    f = subtract_narrow(lam, flux)
    x_blue = 0.5 * (CONT_BLUE[0] + CONT_BLUE[1])
    x_red = 0.5 * (CONT_RED[0] + CONT_RED[1])
    f_blue = np.median(f[(lam >= CONT_BLUE[0]) & (lam <= CONT_BLUE[1])])
    f_red = np.median(f[(lam >= CONT_RED[0]) & (lam <= CONT_RED[1])])
    slope, intercept, *_ = linregress([x_blue, x_red], [f_blue, f_red])
    return f - (slope * lam + intercept)


def bin_mask(lam, k):
    """Pixels of bin k: [lo, hi), last bin [lo, hi] -- no pixel counted twice."""
    if k == nbins - 1:
        return (lam >= bin_lo[k]) & (lam <= bin_hi[k])
    return (lam >= bin_lo[k]) & (lam < bin_hi[k])


def edge_grid(lam, line_only, lo, hi):
    """Bin integration grid: exact edges (interpolated) + interior pixels.

    Bin edges rarely coincide with pixel centers; integrating only the
    pixels inside the bin drops the strips between the edges and the
    outermost pixels -- with only ~4-6 pixels per bin that loses tens of
    percent of the flux. Adding the interpolated edge points makes the
    bins tile the window exactly (adjacent bins share the edge value).
    """
    m = (lam > lo) & (lam < hi)
    w = np.concatenate(([lo], lam[m], [hi]))
    f = np.concatenate(([np.interp(lo, lam, line_only)], line_only[m],
                        [np.interp(hi, lam, line_only)]))
    return w, f


def integrate_bins(lam, flux, err):
    """Simpson-integrate the line profile in every bin (edge-to-edge).

    Returns (flux[nbins], err[nbins]); err = sqrt(sum err_i^2) * dwave over
    the bin's own pixels, dwave outside the root -- matching
    spec_lc_pipeline's integrate_line error convention.
    """
    line_only = line_profile(lam, flux)
    dwave = np.median(np.diff(lam))
    fs, es = np.empty(nbins), np.empty(nbins)
    for k in range(nbins):
        w, f = edge_grid(lam, line_only, bin_lo[k], bin_hi[k])
        if w.size < 3:
            raise ValueError(f"bin {k+1} covers {w.size - 2} pixel(s) -- "
                             "grid too coarse for this binning")
        fs[k] = simpson(f, w)
        es[k] = np.sqrt(np.sum(err[bin_mask(lam, k)] ** 2)) * dwave
    return fs, es


lam0_, flux0_, err0_ = load_spectrum(os.path.join(INPUT_DIR, epoch_names[0]))
npix = [int(bin_mask(lam0_, k).sum()) for k in range(nbins)]
print("pixels per bin:", npix)


## Step 3 — Measure every epoch

One pass over the combined spectra → `bin_flux[epoch, bin]` and the propagated spectral error.

In [ ]:
bin_flux = np.empty((len(epoch_names), nbins))
bin_err_spec = np.empty_like(bin_flux)
for e, name in enumerate(epoch_names):
    lam, flux, err = load_spectrum(os.path.join(INPUT_DIR, name))
    bin_flux[e], bin_err_spec[e] = integrate_bins(lam, flux, err)
print(f"Measured {bin_flux.shape[0]} epochs x {nbins} bins.")


## Step 4 — Per-night scatter from the stacked exposures

Same logic as `spec_lc_pipeline` Step 4 (only the exposures listed in the `combined spec:` header), per bin: `std(ddof=1)/sqrt(N)`; 0 when <2 exposures.

In [ ]:
all_exposures = [f for f in os.listdir(EXPOSURE_DIR)
                 if EXPOSURE_PATTERN in f and not _excluded(f)]
print(f"{len(all_exposures)} per-night exposure files in EXPOSURE_DIR.")

bin_scatter = np.zeros_like(bin_flux)
n_used = []
for e, name in enumerate(epoch_names):
    used_raw = read_used_exposures(name)
    used = [i.split('.')[0] + f'.{EXPOSURE_PATTERN}' for i in used_raw]  # correct to the format used for per night exposures
    if used is None:                                  # no header -> fall back
        used = [f for f in all_exposures if f.startswith(name[:8])]
        print(f"  [warn] {name}: no 'combined spec' header; using all "
              f"{len(used)} same-night exposures.")

    f_bins = []
    for exp in used:
        epath = os.path.join(EXPOSURE_DIR, exp)
        if not os.path.exists(epath):
            print(f"  [warn] {name}: stacked exposure not found: {exp}")
            continue
        lam, flux, err = load_spectrum(epath)
        f_bins.append(integrate_bins(lam, flux, err)[0])

    n_used.append(len(f_bins))
    if len(f_bins) > 1:
        f_bins = np.array(f_bins)
        bin_scatter[e] = (np.nanstd(f_bins, axis=0, ddof=1)
                          / np.sqrt(f_bins.shape[0]))

n_used = np.array(n_used)
print(f"exposures per epoch: min={n_used.min()} max={n_used.max()} "
      f"median={int(np.median(n_used))}")
print(f"median per-night scatter per bin: "
      f"{np.median(bin_scatter[bin_scatter > 0]):.3e}")


## Step 5 — Error budget + write the light curves

`err² = spectral² + scatter² + (syserr·flux)² + narrow_bin²` — identical to `spec_lc_pipeline` Step 5, per bin. One 3-column file per bin (`JD flux err`), bin metadata in the header.

In [ ]:
bin_syserr = syserr[:, None] * bin_flux
bin_err = np.sqrt(bin_err_spec ** 2 + bin_scatter ** 2 + bin_syserr ** 2
                  + narrow_bin_err[None, :] ** 2)

print("Median fractional error budget per bin:")
for k in range(nbins):
    f = bin_flux[:, k]
    print(f"  bin {k+1:2d}: spec {np.median(bin_err_spec[:, k]/f):7.3%} | "
          f"scatter {np.median(bin_scatter[:, k]/f):7.3%} | "
          f"syserr {np.median(bin_syserr[:, k]/f):7.3%} | "
          f"narrow {np.median(narrow_bin_err[k]/f):7.3%} | "
          f"total {np.median(bin_err[:, k]/f):7.3%}")

outfiles = []
for k in range(nbins):
    out = os.path.join(OUTPUT_DIR,
                       f"lc_{LINE_NAME}_{TAG_SPEC}_{SCHEME}_bin{k+1:02d}.txt")
    hdr = (f"{TARGET} {LINE_NAME} velocity bin {k+1}/{nbins} "
           f"({TAG_SPEC} {SCHEME}) | v {bin_vlo[k]:.1f} .. {bin_vhi[k]:.1f} "
           f"km/s | lam {bin_lo[k]:.3f}-{bin_hi[k]:.3f} A | "
           f"flag {bin_flags[k]} | edges: {VELBINS_FILE}\n"
           f"JD  flux(erg/s/cm2)  err")
    np.savetxt(out, np.vstack((jd, bin_flux[:, k], bin_err[:, k])).T,
               header=hdr)
    outfiles.append(out)
    print("wrote", out)


## Step 6 — Consistency checks

1. **Partition**: sum of the bin fluxes vs direct integration of the whole bin window in one piece (Simpson subintervals ≠ whole interval exactly — expect sub-percent agreement).
2. **Against the integrated light curve** `lc_<line>.txt` (different window: `LINE_WINDOW` vs the bin span, so a small constant offset is expected — the ratio should be flat in time).

In [ ]:
# 1 -- partition check (direct integral on the same edge-to-edge grid)
tot_direct = np.empty(len(epoch_names))
for e, name in enumerate(epoch_names):
    lam, flux, err = load_spectrum(os.path.join(INPUT_DIR, name))
    line_only = line_profile(lam, flux)
    w, f = edge_grid(lam, line_only, bin_lo[0], bin_hi[-1])
    tot_direct[e] = simpson(f, w)
tot_bins = bin_flux.sum(axis=1)
rel = np.abs(tot_bins - tot_direct) / tot_direct
print(f"partition check: median |sum(bins)-direct|/direct = {np.median(rel):.2e}, "
      f"max = {rel.max():.2e}")

# 2 -- against the integrated line light curve, if present.
# lc_<line>.txt keeps narrow Hbeta (MAHA convention) and uses LINE_WINDOW,
# so sum(bins) differs by a CONSTANT (narrow Hbeta + window slivers). The
# check is therefore on the DIFFERENCE: it must be flat in time.
lc_int_file = os.path.join(OUTPUT_DIR, f"lc_{LINE_NAME}.txt")
if os.path.exists(lc_int_file):
    lc_int = np.loadtxt(lc_int_file)
    if lc_int.shape[0] == len(jd) and np.allclose(lc_int[:, 0], jd):
        diff = lc_int[:, 1] - tot_bins
        print(f"lc_{LINE_NAME} - sum(bins): median {np.median(diff):.4e}, "
              f"epoch-to-epoch std {np.std(diff, ddof=1):.4e} "
              f"({np.std(diff, ddof=1)/np.median(lc_int[:, 1]):.3%} of the "
              f"line flux; flat difference = consistent)")
    else:
        print(f"[note] {lc_int_file} has a different epoch list -- skipped")
else:
    print(f"[note] {lc_int_file} not found -- run spec_lc_pipeline for check 2")


## Step 7 — Plot all bin light curves

In [ ]:
ncols = 3
nrows = int(np.ceil(nbins / ncols))
t = jd - jd[0]
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.6 * nrows),
                         sharex=True)
for k, ax in enumerate(np.ravel(axes)):
    if k >= nbins:
        ax.axis("off")
        continue
    ax.errorbar(t, bin_flux[:, k], bin_err[:, k], fmt="o", ms=2.5,
                lw=0.8, c="tab:red")
    label = f"bin {k+1}: {bin_vlo[k]:.0f}..{bin_vhi[k]:.0f} km/s"
    if bin_flags[k] != "-":
        label += f" [{bin_flags[k]}]"
    ax.set_title(label, fontsize=9)
    ax.grid(alpha=0.2, lw=0.5)
for ax in np.ravel(axes)[-ncols:]:
    ax.set_xlabel(f"JD - {jd[0]:.0f}")
fig.suptitle(f"{TARGET} {LINE_NAME} -- velocity-bin light curves "
             f"({TAG_SPEC} {SCHEME}, N={nbins})", y=1.0)
fig.tight_layout()
out_pdf = os.path.join(OUTPUT_DIR,
                       f"lc_{LINE_NAME}_{TAG_SPEC}_{SCHEME}_bins.pdf")
fig.savefig(out_pdf, bbox_inches="tight")
print("wrote", out_pdf)
plt.show()
